# Wave Glider OSSE: Spatial-Mean Model Truth

Model truth is WVEL averaged over all model grid points inside the array convex hull,
not just at the centroid.

**To run a different configuration type**, edit `CONFIG_FILES`, `OUTDIR`, and `TITLE` in the
Configuration cell below. Everything else runs identically regardless of array geometry.

In [ ]:
import json
import numpy as np
import os
import sys
from pathlib import Path
import matplotlib.dates as mdates

os.chdir('/home/edavenport/analysis/tpose24-osse')  # pin working directory

sys.path.insert(0, '/home/edavenport/analysis/tpose24-osse')
from osse_tools import (load_model, load_positions, sample_fields,
                        compute_w_planefit, sample_model_w,
                        plot_w_comparison, plot_velocity_map)
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## Configuration

In [ ]:
RUN_DIR   = '/data/SO3/edavenport/tpose24/oct2012_3month_transp_cons'
ITERS     = list(range(36, 26173, 36))

MAX_DEPTH = 120
OUTDIR    = 'spatial_mean/120m'   # output root; one subfolder per config is created here
TITLE     = 'Hexagon array'       # used in figure suptitles

CONFIG_FILES = [
    'configs/hex/d0.25.json',
    'configs/hex/d0.5.json',
    'configs/hex/d0.75.json',
    'configs/hex/d1.0.json',
]

## Load model (once)

In [ ]:
ds = load_model(RUN_DIR, ITERS)
ds = ds.sel(time=slice('2012-10-11', None))  # exclude spin-up

## Run all configurations

In [ ]:
results = {}

for cfg_file in CONFIG_FILES:
    key       = Path(cfg_file).stem
    positions = load_positions(cfg_file)

    uv    = sample_fields(ds, positions, vars=('UVEL', 'VVEL'), max_depth=MAX_DEPTH, dz_obs=2)
    w_est = compute_w_planefit(uv)['w_est']
    print(f'{key}', end='  ')
    w_model = sample_model_w(ds, positions, max_depth=MAX_DEPTH, dz_obs=2, spatial_mean=True)
    bias = w_est - w_model

    results[key] = dict(
        positions=positions, w_est=w_est, w_model=w_model, bias=bias,
        rms=float(np.sqrt((bias**2).mean())),
        mean_bias=float(bias.mean()),
    )
    print(f"RMS={results[key]['rms']:.3e} m/s  "
          f"bias={results[key]['mean_bias']:+.3e} m/s")

## Per-configuration figures

In [ ]:
for key, r in results.items():
    outdir = os.path.join(OUTDIR, key)
    os.makedirs(outdir, exist_ok=True)
    fig = plot_w_comparison(r['w_est'], r['w_model'], point_depth=-50)
    fig.suptitle(f'{TITLE}: {key}', fontsize=13, y=1.01)
    plt.savefig(f'{outdir}/w_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    fig2 = plot_velocity_map(ds, r['positions'], max_depth=MAX_DEPTH)
    fig2.suptitle(f'Depth/time mean velocity | {key}  (0\u2013{MAX_DEPTH} m)',
                  fontsize=12, y=1.01)
    plt.savefig(f'{outdir}/velocity_map.png', dpi=150, bbox_inches='tight')
    plt.show()

## Summary comparison

In [ ]:
keys   = list(results.keys())
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(keys)))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for key, color in zip(keys, colors):
    r = results[key]
    T = r['bias'].time.values
    Z = r['bias'].depth.values
    axes[0, 0].plot(T, r['bias'].mean('depth').values, color=color, lw=1.2, label=key)
    axes[0, 1].plot(r['bias'].mean('time').values, Z,  color=color, lw=1.5, label=key)
    axes[1, 0].plot(T, r['bias'].sel(depth=-50, method='nearest').values,
                    color=color, lw=1.2, label=key)

axes[0, 0].axhline(0, color='k', lw=0.5, ls=':')
axes[0, 0].set_ylabel('Depth-mean bias (m s\u207b\u00b9)')
axes[0, 0].set_title('Depth-mean w error vs time')
axes[0, 0].legend(title='Config', fontsize=9); axes[0, 0].grid(alpha=0.3)
axes[0, 0].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
axes[0, 0].xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0, interval=2))
plt.setp(axes[0, 0].xaxis.get_majorticklabels(), rotation=30, ha='right')

axes[0, 1].axvline(0, color='k', lw=0.7, ls=':')
axes[0, 1].set_xlabel('Time-mean bias (m s\u207b\u00b9)'); axes[0, 1].set_ylabel('Depth (m)')
axes[0, 1].set_title('Time-mean w error vs depth')
axes[0, 1].legend(title='Config', fontsize=9); axes[0, 1].grid(alpha=0.3)

axes[1, 0].axhline(0, color='k', lw=0.5, ls=':')
axes[1, 0].set_ylabel('Bias at 50 m (m s\u207b\u00b9)')
axes[1, 0].set_title('w error at 50 m vs time')
axes[1, 0].legend(title='Config', fontsize=9); axes[1, 0].grid(alpha=0.3)
axes[1, 0].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
axes[1, 0].xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0, interval=2))
plt.setp(axes[1, 0].xaxis.get_majorticklabels(), rotation=30, ha='right')

x_idx    = range(len(keys))
rms_vals = [results[k]['rms']            for k in keys]
mb_vals  = [abs(results[k]['mean_bias']) for k in keys]
axes[1, 1].plot(x_idx, rms_vals, 'o-',  color='C0', lw=1.5, label='RMS error')
axes[1, 1].plot(x_idx, mb_vals,  's--', color='C1', lw=1.5, label='|mean bias|')
axes[1, 1].set_xticks(list(x_idx))
axes[1, 1].set_xticklabels(keys, rotation=30, ha='right')
axes[1, 1].set_ylabel('m s\u207b\u00b9')
axes[1, 1].set_title('Depth-and-time mean error vs config')
axes[1, 1].legend(fontsize=9); axes[1, 1].grid(alpha=0.3)

fig.suptitle(f'{TITLE} (0\u2013{MAX_DEPTH} m)  |  spatial-mean model truth', fontsize=13)
fig.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'summary_error_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## Summary statistics

In [ ]:
header = f"{'Config':>16}  {'RMS (m/s)':>12}  {'Mean bias':>14}  {'w_est std':>12}  {'w_model std':>12}"
print(header); print('-' * len(header))
for key in keys:
    r = results[key]
    print(f"{key:>16}  {r['rms']:>12.3e}  {r['mean_bias']:>+14.3e}  "
          f"{float(r['w_est'].std()):>12.3e}  {float(r['w_model'].std()):>12.3e}")